# Named Entity Recognition

The narrative companion to formalNLP's twenty-second topic. Every number here comes from
`named_entity_recognition.py`, whose harness asserts each claim the page makes. We work on two testbeds:
small chains where we can enumerate every tag sequence and every span subset, and the finance corpus with
**names held out by design**: the taggers train on 30 documents about 18 companies and are tested on a
corpus about the same 18 and on one about 6 companies they never saw.

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/named-entity-recognition").resolve()))
import named_entity_recognition as ner

np.set_printoptions(precision=4, suppress=True)

## 1. Tags encode span sets (Prop 1)

A span set is a set of disjoint typed spans $(s, e, \tau)$. BIO marks a span's first token $\mathrm{B}$-$\tau$ and
the rest $\mathrm{I}$-$\tau$; BIOES marks one-token spans $\mathrm{S}$-$\tau$ and closes longer ones with
$\mathrm{E}$-$\tau$. Each scheme is a bijection between span sets and its *valid* sequences, and validity is a
first-order constraint (a start mask, a transition mask, an end mask), so the valid sequences are counted by a
transfer matrix. Here is the lab's sentence, from the held-out test.

In [ ]:
s = ner.split("unseen")
i = ner.example_sentence()
x, G = s.toks[i], s.spans[i]
print(" ".join(x))
for scheme in ner.SCHEMES:
    tags = ner.tagset(scheme)
    print(f"{scheme:6}", [tags[k] for k in ner.encode(G, len(x), scheme)])
print("span sets on T tokens (8 types) vs all BIO sequences:")
for T in range(1, 6):
    print(T, ner.count_span_sets(T, 8), 17 ** T)
ner.test_prop1_encodings_are_bijections()
print("Prop 1 checks pass")

## 2. BIO against BIOES (Prop 2)

Relabeling $\mathrm{S} \mapsto \mathrm{B}$, $\mathrm{E} \mapsto \mathrm{I}$ turns any BIO chain into a BIOES chain
with the same span law, so BIOES is at least as expressive. With token-local features it is strictly more: on
"a d" (one-token span) and "c d" (two-token span), given $z_1 = \mathrm{B}$ the law of $z_2$ depends on $x_2 = d$
alone, so a BIO chain gives the two gold span sets total probability at most 1. One word of left context closes
the gap.

In [ ]:
ce = ner.prop2_counterexample()
for k, v in ce.items():
    print(f"{k:9} gold probabilities {np.round(v, 6)}  sum {sum(v):.6f}")
ner.test_prop2_bio_chains_are_bioes_chains()
ner.test_prop2_strict_for_token_local_features()
print("Prop 2 checks pass")

## 3. Constrained decoding is conditioning (Thm 1)

Setting invalid starts, transitions, and ends to $-\infty$ multiplies the chain by $\mathbf 1[z \text{ valid}]$,
which factorizes over its edges. Topic 21's recursions then return the best valid sequence, the marginals given
validity, and $\log p(\text{valid} \mid x)$ as a difference of log partition functions.

In [ ]:
ner.test_thm1_constrained_chain_is_the_conditional()
rng = ner._rng(9100)
U, P = rng.normal(size=(1, 4, 5)) * 2, rng.normal(size=(5, 5)) * 2
types = ner.TYPES[:2]
Uc, Pc = ner.constrain(U, P, "bio", types)
tags = ner.tagset("bio", types)
print("unconstrained:", [tags[k] for k in ner.sl.viterbi(U, P)[0][0]])
print("constrained:  ", [tags[k] for k in ner.sl.viterbi(Uc, Pc)[0][0]])
print("p(valid | x) =", float(np.exp(ner.sl.chain_forward(Uc, Pc)[1] - ner.sl.chain_forward(U, P)[1])[0]))

## 4. A CRF on a contextual encoder (Thm 2, Cor 1)

With unaries $U_t = W h_t + b$ on encoder states $h_t$, the gradient of $\log p(z \mid x)$ in $U_t(k)$ is
$\mathbf 1[z_t = k] - \gamma_t(k)$: gold indicators minus node marginals. That vector, times $W^\top$, is what the
encoder's backward pass receives. With the encoder frozen, the objective is Topic 21's on dense features, so it
is strictly concave under an L2 penalty.

In [ ]:
ner.test_thm2_unary_gradient_is_gold_minus_marginals()
ner.test_thm2_chain_rule_into_the_encoder()  # trains Topic 12's encoder on first use (~21 s)
ner.test_cor1_frozen_features_give_a_unique_optimum()
print("Thm 2 and Cor 1 checks pass")

## 5. Span evaluation (Def 4, Prop 3)

Strict matching needs the same boundaries and type; boundary matching ignores the type; type matching needs an
overlap of the same type; partial matching any overlap. Each relaxation can only raise precision, recall, and
F1. Micro F1 is a mixture of the per-type F1s with weights $(|G_\tau| + |\hat P_\tau|)/\sum (|G| + |\hat P|)$,
so a tagger that predicts more spans of a type also gives that type more weight.

In [ ]:
ner.test_prop3_micro_mixture_and_match_ordering()
r = ner.evaluate(ner.GRID["encoder"], "unseen", False)["constrained"]
print("encoder CRF, unseen test, F1 by match:", {k: round(v, 6) for k, v in r["schemes"].items()})
print("mixture weights:", {k: round(v, 4) for k, v in r["weights"].items()})

## 6. F1-optimal span decoding (Prop 4, Thm 3)

A span's marginal is a forward message, a product along the span, and a backward message. Maximizing F1 of
expected counts, $2\sum_{\hat S} q / (|\hat S| + m)$, is a fractional program: Dinkelbach's iteration solves a
weighted-interval problem with weights $2q - F$ until $F$ stops rising, and the optimal set thresholds the
marginals at $F^*/2$. With disjoint candidates this is Lipton, Elkan, and Naryanaswamy's threshold.

In [ ]:
ner.test_prop4_span_marginals_match_brute_force()
ner.test_thm3_dinkelbach_is_the_f1_optimum()
ner.test_thm3_disjoint_candidates_threshold_at_half_f()
print("Prop 4 and Thm 3 checks pass")

## 7. Names held out by design

`fc.split_roster()` holds out one company per sector, with its subsidiaries, officers, and flagship product.
Every ORG, PERSON, and PRODUCT mention of the held-out test contains a word training never shows; most of those
words were nonetheless in Topic 12's pretraining text, the regime a pretrained encoder is in.

In [ ]:
print("held out:", ner.HELD_ROSTER)
for t in ("train", *ner.TESTS):
    sp_ = ner.split(t)
    print(f"{t:7} sentences {len(sp_.toks)}  tokens {sum(map(len, sp_.toks))}  spans {sum(map(len, sp_.spans))}")
print(ner.pretraining_exposure())
ner.test_held_out_design()

## 8. Who recovers an unseen name

Strict F1 on both tests for every tagger (constrained Viterbi). Encoder taggers are read twice: with the
encoder's own vocabulary ("ids") and with every word unseen in training entering as [UNK] ("unk").

In [ ]:
for name, m in ner.GRID.items():
    for t, u in ner.readings(m):
        c = ner.evaluate(m, t, u)["constrained"]
        print(f"{name:32} {ner.reading_key(t, u):11} F1 {c['f1']:.6f}  P {c['p']:.4f}  R {c['r']:.4f}  ORG {c['types']['ORG']:.4f}")
ner.test_the_word_crf_memorizes_names()
ner.test_case_is_most_of_the_signal()
ner.test_encoder_context()

## 9. Dropout must delete, not substitute (Prop 5)

Rewriting dropped entity tokens as `<unk>` makes `<unk>` a feature that fires only on entity tokens in training.
At the penalized optimum such a feature has a negative weight for O (Prop 5), so at test every unseen word, name
or not, is pushed toward an entity. Deleting the token's identity features adds no feature to learn that from.

In [ ]:
ner.test_prop5_a_feature_seen_only_on_entities_argues_against_o()
sw = ner.dropout_sweep()
print("p:", ner.DROPOUT_SWEEP)
for how in sw:
    for t in ner.TESTS:
        print(f"{how:10} {t:6} F1 {np.round(sw[how][t]['f1'], 6)}  P {np.round(sw[how][t]['p'], 4)}")
ner.test_dropout_recipes()

## 10. Heads, constraints, and BIOES

On frozen states a per-token softmax decodes invalid BIO sequences; Thm 1's constraint removes them, but the CRF's
transitions are worth far more. BIOES buys nothing here: the word features already see the neighbors (Prop 2's
equality case).

In [ ]:
ner.test_crf_head_vs_softmax_head()
ner.test_bioes_buys_nothing_here()
for t in ner.TESTS:
    s_ = ner.evaluate(ner.GRID["softmax on encoder"], t, False)
    print(t, "softmax raw", round(s_["raw"]["f1"], 6), "constrained", round(s_["constrained"]["f1"], 6),
          "invalid share", round(s_["invalid"], 6))

## 11. Thm 3 on the finance taggers

F1-optimal decoding is optimal for the model's own beliefs. It rescues the miscalibrated substitution tagger and
leaves the headline tagger where Viterbi put it; neither model's $F^*$ is its realized F1.

In [ ]:
ner.test_overlapping_spans_are_exclusive()
for name, u in ner.F1_MODELS:
    for t in ner.TESTS:
        c = ner.threshold_curve(name, t, u)
        print(f"{name:24} {t:6} F* {c['F_star']:.6f}  Viterbi {c['viterbi']['f1']:.6f}  F1-decoded {c['f1_decoded']['f1']:.6f}")
ner.test_thm3_on_the_finance_taggers()
ner.test_prop3_on_the_finance_taggers()

## Harness

Every assert in the module, then the summary the page quotes.

In [ ]:
ner._run_harness()